In [1]:
import os
import gc
import time
import json
import ctypes
from datetime import datetime

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from torchvision import transforms, models
from sklearn.metrics import roc_auc_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
GPU: NVIDIA GeForce RTX 5060 Laptop GPU


In [2]:
# Paths
RAW_DIR = os.path.join('..', 'data', 'raw')
PROCESSED_DIR = os.path.join('..', 'data', 'processed')
TEST_DIR = os.path.join('..', 'data', 'test')
MODELS_DIR = os.path.join('..', 'models', 'noise_experiments')
RESULTS_DIR = os.path.join('..', 'results', 'noise_experiments')
LOG_DIR = os.path.join('..', 'results', 'logs')
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

TARGET_PATHOLOGIES = [
    'Cardiomegaly', 'Edema', 'Consolidation',
    'Atelectasis', 'Pleural Effusion'
]

# Training hyperparameters (same as baseline)
TRAIN_CONFIG = {
    'batch_size': 32,
    'num_workers': 0,
    'learning_rate': 1e-4,
    'num_epochs': 5,
    'image_size': 224,
    'weight_decay': 1e-5,
    'early_stopping_patience': 2,
}

# Experiment matrix: (noise_type, noise_rate) pairs
NOISE_CONFIGS = [
    ('symmetric', 0.05), ('symmetric', 0.10),
    ('symmetric', 0.20), ('symmetric', 0.40),
    ('asymmetric', 0.05), ('asymmetric', 0.10),
    ('asymmetric', 0.20), ('asymmetric', 0.40),
]

SEEDS = [132, 456, 789]

# Cooldown between experiments (thermal protection)
COOLDOWN_SECONDS = 180  # 3 minutes

print(f"Noise configurations: {len(NOISE_CONFIGS)}")
print(f"Seeds per config: {len(SEEDS)}")
print(f"Total experiments: {len(NOISE_CONFIGS) * len(SEEDS)}")
print(f"Estimated time: {len(NOISE_CONFIGS) * len(SEEDS) * 90 / 60:.1f} hours + cooldowns")

Noise configurations: 8
Seeds per config: 3
Total experiments: 24
Estimated time: 36.0 hours + cooldowns


In [3]:
# Windows API constants
ES_CONTINUOUS = 0x80000000
ES_SYSTEM_REQUIRED = 0x00000001
ES_DISPLAY_REQUIRED = 0x00000002

def prevent_sleep():
    """Keep Windows awake during training."""
    try:
        ctypes.windll.kernel32.SetThreadExecutionState(
            ES_CONTINUOUS | ES_SYSTEM_REQUIRED | ES_DISPLAY_REQUIRED
        )
        print("Sleep prevention ENABLED")
    except Exception as e:
        print(f"Could not prevent sleep: {e}")

def allow_sleep():
    """Restore normal sleep behavior."""
    try:
        ctypes.windll.kernel32.SetThreadExecutionState(ES_CONTINUOUS)
        print("Sleep prevention DISABLED")
    except Exception as e:
        print(f"Could not restore sleep: {e}")

prevent_sleep()

Sleep prevention ENABLED


In [4]:
class CheXpertDataset(Dataset):
    def __init__(self, csv_path, data_dir, pathologies, transform=None):
        self.df = pd.read_csv(csv_path)
        self.data_dir = data_dir
        self.pathologies = pathologies
        self.transform = transform
        self.labels = self.df[pathologies].values.astype(np.float32)
        self.paths = self.df['Path'].values
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        img_path = os.path.join(self.data_dir, self.paths[idx])
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, torch.from_numpy(self.labels[idx])

train_transform = transforms.Compose([
    transforms.Resize((TRAIN_CONFIG['image_size'], TRAIN_CONFIG['image_size'])),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

eval_transform = transforms.Compose([
    transforms.Resize((TRAIN_CONFIG['image_size'], TRAIN_CONFIG['image_size'])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Val and test datasets are shared across all experiments
val_dataset = CheXpertDataset(
    csv_path=os.path.join(PROCESSED_DIR, 'val_clean.csv'),
    data_dir=RAW_DIR,
    pathologies=TARGET_PATHOLOGIES,
    transform=eval_transform
)
test_dataset = CheXpertDataset(
    csv_path=os.path.join(PROCESSED_DIR, 'test_official.csv'),
    data_dir=TEST_DIR,
    pathologies=TARGET_PATHOLOGIES,
    transform=eval_transform
)

val_loader = DataLoader(val_dataset, batch_size=TRAIN_CONFIG['batch_size'],
                        shuffle=False, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=TRAIN_CONFIG['batch_size'],
                         shuffle=False, num_workers=0, pin_memory=True)

print(f"Val:  {len(val_dataset)} images")
print(f"Test: {len(test_dataset)} images")

Val:  202 images
Test: 518 images


In [5]:
def build_densenet121(num_classes, pretrained=True):
    weights = models.DenseNet121_Weights.IMAGENET1K_V1 if pretrained else None
    model = models.densenet121(weights=weights)
    model.classifier = nn.Linear(model.classifier.in_features, num_classes)
    return model


def train_one_epoch(model, loader, criterion, optimizer, scaler, device):
    model.train()
    total_loss = 0.0
    n_batches = 0
    start = time.time()
    
    for i, (images, labels) in enumerate(loader):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        
        if scaler is not None:
            with autocast('cuda'):
                logits = model(images)
                loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
        
        total_loss += loss.item()
        n_batches += 1
        
        if (i + 1) % 200 == 0:
            elapsed = time.time() - start
            eta = elapsed / (i + 1) * (len(loader) - i - 1)
            print(f"    Batch {i+1:>5}/{len(loader)} | loss={loss.item():.4f} | "
                  f"elapsed={elapsed/60:.1f}min | ETA={eta/60:.1f}min")
    
    return total_loss / n_batches


@torch.no_grad()
def evaluate(model, loader, criterion, device, pathologies):
    model.eval()
    total_loss = 0.0
    all_logits = []
    all_labels = []
    
    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        with autocast('cuda', enabled=device.type == 'cuda'):
            logits = model(images)
            loss = criterion(logits, labels)
        total_loss += loss.item()
        all_logits.append(logits.float().cpu().numpy())
        all_labels.append(labels.cpu().numpy())
    
    all_logits = np.concatenate(all_logits)
    all_labels = np.concatenate(all_labels)
    all_probs = 1 / (1 + np.exp(-all_logits))
    
    aucs = {}
    for i, path in enumerate(pathologies):
        if len(np.unique(all_labels[:, i])) > 1:
            aucs[path] = float(roc_auc_score(all_labels[:, i], all_probs[:, i]))
        else:
            aucs[path] = float('nan')
    
    mean_auc = float(np.nanmean(list(aucs.values())))
    return total_loss / len(loader), mean_auc, aucs

In [6]:
def log_message(msg):
    """Print + append to log file."""
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    line = f"[{timestamp}] {msg}"
    print(line)
    with open(os.path.join(LOG_DIR, 'noise_experiments.log'), 'a', encoding='utf-8') as f:
        f.write(line + '\n')


def run_experiment(noise_type, noise_rate, seed, train_config, verbose=True):
    """Train one noise experiment. Returns dict with all results."""
    exp_name = f"{noise_type}_{int(noise_rate*100):02d}_seed{seed}"
    exp_start = time.time()
    
    log_message(f"START: {exp_name}")
    
    # Set seeds for reproducibility
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    
    # Build training dataset with noisy labels
    train_csv = os.path.join(PROCESSED_DIR, f'train_{noise_type}_{int(noise_rate*100):02d}.csv')
    train_dataset = CheXpertDataset(
        csv_path=train_csv, data_dir=RAW_DIR,
        pathologies=TARGET_PATHOLOGIES, transform=train_transform
    )
    train_loader = DataLoader(
        train_dataset, batch_size=train_config['batch_size'],
        shuffle=True, num_workers=0, pin_memory=True
    )
    
    # Fresh model + optimizer
    model = build_densenet121(len(TARGET_PATHOLOGIES), pretrained=True).to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.AdamW(
        model.parameters(),
        lr=train_config['learning_rate'],
        weight_decay=train_config['weight_decay']
    )
    scaler = GradScaler('cuda') if device.type == 'cuda' else None
    
    # Checkpoint paths
    ckpt_path = os.path.join(MODELS_DIR, f'{exp_name}_last.pt')
    best_path = os.path.join(MODELS_DIR, f'{exp_name}_best.pt')
    
    history = {'train_loss': [], 'val_loss': [], 'val_auc': [], 'per_pathology_auc': []}
    best_val_auc = 0.0
    epochs_no_improve = 0
    start_epoch = 1
    
    # Resume from checkpoint if available
    if os.path.exists(ckpt_path):
        try:
            ckpt = torch.load(ckpt_path, weights_only=False)
            model.load_state_dict(ckpt['model_state_dict'])
            optimizer.load_state_dict(ckpt['optimizer_state_dict'])
            start_epoch = ckpt['epoch'] + 1
            history = ckpt.get('history', history)
            best_val_auc = ckpt.get('best_val_auc', 0.0)
            log_message(f"  Resumed from epoch {ckpt['epoch']}, best_val_auc={best_val_auc:.4f}")
        except Exception as e:
            log_message(f"  Failed to resume: {e}. Starting fresh.")
            start_epoch = 1
    
    # Training loop
    for epoch in range(start_epoch, train_config['num_epochs'] + 1):
        epoch_start = time.time()
        if verbose:
            print(f"  Epoch {epoch}/{train_config['num_epochs']}")
        
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, scaler, device)
        val_loss, val_auc, per_path_auc = evaluate(model, val_loader, criterion, device, TARGET_PATHOLOGIES)
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_auc'].append(val_auc)
        history['per_pathology_auc'].append(per_path_auc)
        
        epoch_time = time.time() - epoch_start
        log_message(f"  Epoch {epoch}: train_loss={train_loss:.4f}, val_auc={val_auc:.4f}, time={epoch_time/60:.1f}min")
        
        # Save per-epoch checkpoint (for resume)
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'history': history,
            'best_val_auc': best_val_auc,
        }, ckpt_path)
        
        # Save best model separately
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            epochs_no_improve = 0
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'val_auc': val_auc,
                'per_pathology_auc': per_path_auc,
            }, best_path)
            log_message(f"    New best (AUC={val_auc:.4f})")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= train_config['early_stopping_patience']:
                log_message(f"  Early stopping at epoch {epoch}")
                break
    
    # Load best model and evaluate on official test set
    log_message(f"  Evaluating best model on official test set...")
    best_ckpt = torch.load(best_path, weights_only=False)
    model.load_state_dict(best_ckpt['model_state_dict'])
    test_loss, test_auc, test_per_path = evaluate(model, test_loader, criterion, device, TARGET_PATHOLOGIES)
    
    exp_time = time.time() - exp_start
    
    # Save complete results
    results = {
        'experiment_name': exp_name,
        'noise_type': noise_type,
        'noise_rate': noise_rate,
        'seed': seed,
        'best_epoch': best_ckpt['epoch'],
        'best_val_auc': float(best_val_auc),
        'best_val_per_pathology_auc': {k: float(v) for k, v in best_ckpt['per_pathology_auc'].items()},
        'test_mean_auc': float(test_auc),
        'test_per_pathology_auc': {k: float(v) for k, v in test_per_path.items()},
        'test_loss': float(test_loss),
        'history': history,
        'total_time_minutes': exp_time / 60,
        'config': train_config,
    }
    
    results_path = os.path.join(RESULTS_DIR, f'{exp_name}.json')
    with open(results_path, 'w') as f:
        json.dump(results, f, indent=2)
    
    log_message(f"DONE: {exp_name} | test_auc={test_auc:.4f} | time={exp_time/60:.1f}min")
    
    # Cleanup GPU memory
    del model, optimizer, scaler, train_dataset, train_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    return results

In [7]:
# Check completed experiments
completed = sorted([f.replace('.json', '') for f in os.listdir(RESULTS_DIR) if f.endswith('.json')])
print(f"Completed experiments ({len(completed)}):")
for name in completed:
    print(f"  ✓ {name}")

# Check checkpoint files
print(f"\nCheckpoint files:")
for name in sorted(os.listdir(MODELS_DIR)):
    fpath = os.path.join(MODELS_DIR, name)
    size_mb = os.path.getsize(fpath) / 1e6
    mtime = datetime.fromtimestamp(os.path.getmtime(fpath)).strftime('%Y-%m-%d %H:%M')
    print(f"  {name:50s} {size_mb:>6.1f}MB  {mtime}")

Completed experiments (10):
  ✓ symmetric_05_seed132
  ✓ symmetric_05_seed456
  ✓ symmetric_05_seed789
  ✓ symmetric_10_seed132
  ✓ symmetric_10_seed456
  ✓ symmetric_10_seed789
  ✓ symmetric_20_seed132
  ✓ symmetric_20_seed456
  ✓ symmetric_20_seed789
  ✓ symmetric_40_seed132

Checkpoint files:
  symmetric_05_seed132_best.pt                         28.5MB  2026-07-28 23:24
  symmetric_05_seed132_last.pt                         84.5MB  2026-07-28 23:42
  symmetric_05_seed456_best.pt                         28.5MB  2026-07-29 00:22
  symmetric_05_seed456_last.pt                         84.5MB  2026-07-29 00:59
  symmetric_05_seed789_best.pt                         28.5MB  2026-07-29 02:16
  symmetric_05_seed789_last.pt                         84.5MB  2026-07-29 02:35
  symmetric_10_seed132_best.pt                         28.5MB  2026-07-29 03:52
  symmetric_10_seed132_last.pt                         84.5MB  2026-07-29 04:11
  symmetric_10_seed456_best.pt                         28.5MB  

In [8]:
all_experiments = []
for noise_type, noise_rate in NOISE_CONFIGS:
    for seed in SEEDS:
        exp_name = f"{noise_type}_{int(noise_rate*100):02d}_seed{seed}"
        all_experiments.append((noise_type, noise_rate, seed, exp_name))

completed = []
pending = []
for noise_type, noise_rate, seed, exp_name in all_experiments:
    results_path = os.path.join(RESULTS_DIR, f'{exp_name}.json')
    if os.path.exists(results_path):
        completed.append(exp_name)
    else:
        pending.append((noise_type, noise_rate, seed, exp_name))

print(f"Total experiments: {len(all_experiments)}")
print(f"Already completed: {len(completed)}")
print(f"Pending:           {len(pending)}")
if completed:
    print(f"\nCompleted experiments:")
    for name in completed:
        print(f"  ✓ {name}")
if pending:
    print(f"\nPending experiments:")
    for _, _, _, name in pending:
        print(f"  ○ {name}")

print(f"\nEstimated remaining time: {len(pending) * 90 / 60:.1f} hours + cooldowns")

Total experiments: 24
Already completed: 10
Pending:           14

Completed experiments:
  ✓ symmetric_05_seed132
  ✓ symmetric_05_seed456
  ✓ symmetric_05_seed789
  ✓ symmetric_10_seed132
  ✓ symmetric_10_seed456
  ✓ symmetric_10_seed789
  ✓ symmetric_20_seed132
  ✓ symmetric_20_seed456
  ✓ symmetric_20_seed789
  ✓ symmetric_40_seed132

Pending experiments:
  ○ symmetric_40_seed456
  ○ symmetric_40_seed789
  ○ asymmetric_05_seed132
  ○ asymmetric_05_seed456
  ○ asymmetric_05_seed789
  ○ asymmetric_10_seed132
  ○ asymmetric_10_seed456
  ○ asymmetric_10_seed789
  ○ asymmetric_20_seed132
  ○ asymmetric_20_seed456
  ○ asymmetric_20_seed789
  ○ asymmetric_40_seed132
  ○ asymmetric_40_seed456
  ○ asymmetric_40_seed789

Estimated remaining time: 21.0 hours + cooldowns


In [9]:
session_start = time.time()
log_message(f"=== SESSION START ===")
log_message(f"Pending experiments: {len(pending)}")

for i, (noise_type, noise_rate, seed, exp_name) in enumerate(pending):
    print("\n" + "=" * 80)
    print(f"EXPERIMENT {i+1}/{len(pending)}: {exp_name}")
    print("=" * 80)
    
    try:
        results = run_experiment(noise_type, noise_rate, seed, TRAIN_CONFIG)
        
        # Print summary
        print(f"\n  ✓ Completed: test_auc={results['test_mean_auc']:.4f}")
        for path, auc in results['test_per_pathology_auc'].items():
            print(f"      {path:20s}: {auc:.4f}")
        
    except Exception as e:
        log_message(f"ERROR in {exp_name}: {e}")
        print(f"\n  ✗ FAILED: {e}")
        print(f"  Checkpoint saved. Re-run this cell to resume.")
        # Don't crash the whole loop - move to next experiment
        continue
    
    # Cooldown between experiments (except after last one)
    if i < len(pending) - 1:
        print(f"\n  Cooling down for {COOLDOWN_SECONDS}s before next experiment...")
        for remaining in range(COOLDOWN_SECONDS, 0, -30):
            print(f"    {remaining}s remaining", end='\r')
            time.sleep(min(30, remaining))
        print("    Cooldown complete.       ")

session_time = time.time() - session_start
log_message(f"=== SESSION END === Total time: {session_time/3600:.2f} hours")

print("\n" + "=" * 80)
print(f"SESSION COMPLETE")
print(f"Total time: {session_time/3600:.2f} hours")
print("=" * 80)

[2026-07-30 14:41:32] === SESSION START ===
[2026-07-30 14:41:32] Pending experiments: 14

EXPERIMENT 1/14: symmetric_40_seed456
[2026-07-30 14:41:32] START: symmetric_40_seed456
[2026-07-30 14:41:33]   Resumed from epoch 4, best_val_auc=0.8285
  Epoch 5/5
    Batch   200/5970 | loss=0.6898 | elapsed=0.6min | ETA=17.7min
    Batch   400/5970 | loss=0.7037 | elapsed=1.2min | ETA=16.6min
    Batch   600/5970 | loss=0.6914 | elapsed=1.8min | ETA=15.8min
    Batch   800/5970 | loss=0.6910 | elapsed=2.3min | ETA=15.1min
    Batch  1000/5970 | loss=0.6905 | elapsed=2.9min | ETA=14.5min
    Batch  1200/5970 | loss=0.6874 | elapsed=3.5min | ETA=13.8min
    Batch  1400/5970 | loss=0.6994 | elapsed=4.0min | ETA=13.2min
    Batch  1600/5970 | loss=0.6812 | elapsed=4.6min | ETA=12.6min
    Batch  1800/5970 | loss=0.6919 | elapsed=5.2min | ETA=12.0min
    Batch  2000/5970 | loss=0.6786 | elapsed=5.8min | ETA=11.4min
    Batch  2200/5970 | loss=0.6854 | elapsed=6.3min | ETA=10.8min
    Batch  2400/5

In [10]:
# Load all completed results
rows = []
for fname in sorted(os.listdir(RESULTS_DIR)):
    if not fname.endswith('.json'):
        continue
    with open(os.path.join(RESULTS_DIR, fname)) as f:
        r = json.load(f)
    row = {
        'experiment': r['experiment_name'],
        'noise_type': r['noise_type'],
        'noise_rate': r['noise_rate'],
        'seed': r['seed'],
        'test_mean_auc': r['test_mean_auc'],
    }
    for path, auc in r['test_per_pathology_auc'].items():
        row[f'test_{path.replace(" ", "_")}'] = auc
    rows.append(row)

if rows:
    results_df = pd.DataFrame(rows)
    print(f"Loaded {len(results_df)} experiment results")
    
    # Aggregate: mean across seeds per (noise_type, noise_rate)
    agg_cols = [c for c in results_df.columns if c.startswith('test_')]
    aggregated = results_df.groupby(['noise_type', 'noise_rate'])[agg_cols].agg(['mean', 'std']).round(4)
    
    # Save aggregated CSV
    results_df.to_csv(os.path.join(RESULTS_DIR, 'all_results.csv'), index=False)
    aggregated.to_csv(os.path.join(RESULTS_DIR, 'aggregated_results.csv'))
    
    print(f"\nSaved: {os.path.join(RESULTS_DIR, 'all_results.csv')}")
    print(f"Saved: {os.path.join(RESULTS_DIR, 'aggregated_results.csv')}")
    
    # Display summary
    print("\n=== Test Mean AUC by config (mean +- std across seeds) ===")
    summary = results_df.groupby(['noise_type', 'noise_rate'])['test_mean_auc'].agg(['mean', 'std', 'count']).round(4)
    print(summary)
else:
    print("No results found yet.")

Loaded 24 experiment results

Saved: ..\results\noise_experiments\all_results.csv
Saved: ..\results\noise_experiments\aggregated_results.csv

=== Test Mean AUC by config (mean +- std across seeds) ===
                         mean     std  count
noise_type noise_rate                       
asymmetric 0.05        0.8791  0.0106      3
           0.10        0.8781  0.0061      3
           0.20        0.8681  0.0055      3
           0.40        0.8489  0.0086      3
symmetric  0.05        0.8791  0.0036      3
           0.10        0.8822  0.0057      3
           0.20        0.8769  0.0057      3
           0.40        0.8339  0.0172      3


In [11]:
# Only run this AFTER all experiments are complete
allow_sleep()

Sleep prevention DISABLED
